# Inference on New Vestibular Schwannoma Cases

Run one all-data model or an explicitly declared soft-voting ensemble and save NIfTI tumor masks. Preprocessing and decoding are read from each Safetensors artifact, and all members must carry the same inference contract.

## 1. Environment setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from fastMONAI.vision_all import (
    MedImage, MedMask, evaluate_segmentations, med_img_reader,
    patch_inference, prediction_filename,
)
from fastMONAI.vision_plot import find_max_slice, show_mask_overlay

start = Path.cwd().resolve()
if (start / "notebooks").is_dir() and (start / "README.md").is_file():
    PROJECT_ROOT = start
elif start.name == "notebooks" and (start.parent / "README.md").is_file():
    PROJECT_ROOT = start.parent
else:
    raise FileNotFoundError("Start Jupyter from vestibular_schwannoma or its notebooks directory")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from workflow.inference import load_predictor_set, resolve_model_artifacts


## 2. Configuration

In [ ]:
DEPLOYMENT_MODE = "ensemble"  # "single" or "ensemble"
ARTIFACT_ROLE = "best"        # use "final" for an all-data run

MEMBER_RUN_IDS = {
    # "fold_1": "<mlflow-run-id>",
    # "fold_2": "<mlflow-run-id>",
}
LOCAL_MODEL_ARTIFACTS = {
    # "all_data": "/path/to/final_model.safetensors",
}

NEW_CASES = [
    "/path/to/case_t1.nii.gz",
]
OUTPUT_DIR = "inference_predictions"

USE_TTA = True
USE_AMP = True

print(f"Declared mode: {DEPLOYMENT_MODE}")
print(f"Cases to segment: {len(NEW_CASES)}")

### Resolve the declared model artifacts

In [ ]:
if DEPLOYMENT_MODE not in {"single", "ensemble"}:
    raise ValueError(f"Unknown DEPLOYMENT_MODE: {DEPLOYMENT_MODE!r}")
if ARTIFACT_ROLE not in {"best", "final"}:
    raise ValueError("ARTIFACT_ROLE must be 'best' or 'final'")
if not NEW_CASES:
    raise ValueError("NEW_CASES must contain at least one image")

missing_cases = [path for path in NEW_CASES if not Path(path).is_file()]
if missing_cases:
    raise FileNotFoundError(f"Input images not found: {missing_cases}")

model_artifacts = resolve_model_artifacts(
    member_run_ids=MEMBER_RUN_IDS,
    local_model_artifacts=LOCAL_MODEL_ARTIFACTS,
    artifact_role=ARTIFACT_ROLE,
)
print("Declared artifacts:")
for member, artifact in model_artifacts.items():
    print(f"  {member}: {artifact}")


## 3. Validate the inference contract and load models


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
loaded = load_predictor_set(
    model_artifacts,
    mode=DEPLOYMENT_MODE,
    artifact_role=ARTIFACT_ROLE,
    device=device,
    expected_run_ids=MEMBER_RUN_IDS or None,
)
patch_config = loaded.patch_config
print("Inference config source: Safetensors metadata")
print(patch_config)


## 4. Inspect the loaded predictor set


In [ ]:
predictor = loaded.predictor
predictors = loaded.models
for member, artifact in loaded.artifacts.items():
    print(f"Loaded {member}: {artifact}")
print(f"Ready: {DEPLOYMENT_MODE} deployment with {len(predictors)} model(s) on {device}.")


## 5. Run inference

`patch_inference` applies the embedded preprocessing and decoding contract and soft-votes declared ensembles. Predictions are saved in `OUTPUT_DIR`.

`USE_TTA` averages eight flips; `USE_AMP` enables CUDA bfloat16 inference.

In [ ]:
predictions = patch_inference(
    learner=predictor,
    config=patch_config,
    file_paths=NEW_CASES,
    save_dir=OUTPUT_DIR,
    progress=True,
    tta=USE_TTA,
    amp=USE_AMP,
)

label = (
    "one all-data model"
    if DEPLOYMENT_MODE == "single"
    else f"soft-vote ensemble of {len(predictors)} declared models"
)
print()
print(f"Wrote {len(predictions)} prediction(s) to {OUTPUT_DIR}/ ({label}).")
for path in NEW_CASES:
    print(f"  {path}  ->  {OUTPUT_DIR}/{prediction_filename(path)}")


## 6. Visualize a prediction

In [ ]:
idx = 0
img_fn = NEW_CASES[idx]
pred_fn = Path(OUTPUT_DIR) / prediction_filename(img_fn)

img = MedImage.create(img_fn)
pred_mask = MedMask.create(pred_fn)
org_img, _, _ = med_img_reader(img_fn, only_tensor=False)
disp_spacing = org_img.spacing

plane = 2  # 0 = sagittal, 1 = coronal, 2 = axial
sl = int(find_max_slice(pred_mask.data[0].cpu().numpy(), plane))
print(f"Predicted foreground voxels: {int(pred_mask.data.sum())}")
print(f"Showing plane={plane} (axial), slice={sl}, spacing={disp_spacing}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
img.show(ctx=axes[0], anatomical_plane=plane, slice_index=sl, voxel_size=disp_spacing)
axes[0].set_title("Input T1")
pred_mask.show(ctx=axes[1], anatomical_plane=plane, slice_index=sl, voxel_size=disp_spacing)
axes[1].set_title("Predicted mask")
show_mask_overlay(
    img,
    pred_mask,
    ctx=axes[2],
    anatomical_plane=plane,
    slice_index=sl,
    voxel_size=disp_spacing,
    title="Overlay",
)
plt.tight_layout()
plt.show()


## 7. Optional: validate against ground truth

Set `GT_PATH` to score the first prediction with the same metrics as notebook 01.

In [ ]:
GT_PATH = None

if GT_PATH and not Path(GT_PATH).is_file():
    raise FileNotFoundError(f"Ground-truth mask not found: {GT_PATH}")
if GT_PATH:
    row = evaluate_segmentations([predictions[0]], [GT_PATH]).iloc[0]
    print(f"DSC:          {row['dsc']:.4f}")
    print(f"Sensitivity:  {row['sensitivity']:.4f}")
    print(f"Precision:    {row['precision']:.4f}")
    print(f"LDR:          {row['ldr']:.4f}")
    print(f"Signed RVE:   {row['rve']:.4f}")
    print(f"ASSD (mm):    {row['assd_mm']}")
    print(f"HD95 (mm):    {row['hd95_mm']}")
    print(f"NSD tau=1mm:  {row['nsd_tau1.0_mm']}")
    print(f"Spacing (mm): {row['spacing_mm']}  | status: {row['surface_status']}")
else:
    print("GT_PATH is None; skipping ground-truth evaluation.")

## 8. SegMamba backend

SegMamba artifacts use the backend recorded in their registered model specification. Backend substitution is unsupported.